4. Column Operations
│   ├── select()
│   ├── col()
│   ├── alias()
│   ├── withColumn()
│   ├── withColumnRenamed()
│   ├── drop()
│   └── cast()

select() 
- select() is used to choose one or more columns from dataframe 
- return a new dataframe 
- transformation 
│
├── 5. Filtering
│   ├── filter()
│   ├── where()
│   ├── AND / OR / NOT
│   ├── isin()
│   ├── between()
│   └── NULL handling
│
├── 6. Conditional Logic
│   ├── when()
│   ├── otherwise()
│   └── Multiple conditions
│
├── 7. Aggregations
│   ├── groupBy()
│   ├── count()
│   ├── sum()
│   ├── avg()
│   ├── min()
│   ├── max()
│   └── agg()
│
├── 8. Joins
│   ├── inner
│   ├── left
│   ├── right
│   ├── full
│   ├── left_semi
│   ├── left_anti
│   ├── cross
│   └── broadcast join
│
├── 9. Sorting & Duplicates
│   ├── orderBy()
│   ├── sort()
│   ├── distinct()
│   └── dropDuplicates()
│
├── 10. Null Handling
│   ├── isNull()
│   ├── isNotNull()
│   ├── dropna()
│   ├── fillna()
│   └── coalesce()
│
├── 11. String / Date Functions
│
├── 12. Window Functions
│   ├── row_number()
│   ├── rank()
│   ├── dense_rank()
│   ├── lag()
│   └── lead()
│

# PYSPARK DATAFRAME – DETAILED THEORY NOTES

# 3. UNDERSTANDING DATAFRAMES

A Spark DataFrame is a distributed collection of data organized into named columns.

It is similar to:

- A table in a relational database
- A Pandas DataFrame
- An Excel table

But unlike Pandas, a Spark DataFrame is designed to process very large datasets distributed across multiple machines.

A DataFrame contains two major things:

```text
Data
+
Schema
```

Example:

```text
+-----------+-------+------+
|customer_id|product|amount|
+-----------+-------+------+
|C101       |Laptop |50000 |
|C102       |Mobile |20000 |
|C103       |TV     |30000 |
+-----------+-------+------+
```

Schema:

```text
customer_id → string
product     → string
amount      → long
```

Spark provides multiple functions and properties to understand the contents and structure of a DataFrame.

---

# 3.1 `show()`

## Definition

`show()` is used to display DataFrame records in tabular format.

It is mainly used during:

- Development
- Debugging
- Data validation
- Testing transformations
- Checking output

---

## Syntax

```python
df.show()
```

By default Spark displays:

```text
20 rows
```

Example:

```python
df.show(5)
```

Displays only 5 rows.

---

## Complete Syntax

```python
df.show(
    n=20,
    truncate=True,
    vertical=False
)
```

---

## Parameter 1: `n`

Controls how many rows are displayed.

Example:

```python
df.show(10)
```

Meaning:

```text
Display 10 rows
```

---

## Parameter 2: `truncate`

By default, Spark may truncate long string values.

Example:

```python
df.show()
```

A long value may appear as:

```text
Customer contacted...
```

To display complete values:

```python
df.show(
    truncate=False
)
```

This is very useful when working with:

- JSON
- Long descriptions
- Addresses
- API responses
- Error messages
- Nested data

---

## Parameter 3: `vertical`

Normally Spark displays data horizontally.

Example:

```text
+---+------+------+
|id |name  |salary|
+---+------+------+
```

If the DataFrame contains many columns, horizontal output becomes difficult to read.

Use:

```python
df.show(
    vertical=True
)
```

Output conceptually:

```text
-RECORD 0-------
 id     | 101
 name   | Anuj
 salary | 85000
```

This is useful for very wide DataFrames.

---

## Example

```python
data = [
    (101, "Anuj", 85000),
    (102, "Rahul", 72000),
    (103, "Neha", 91000)
]

df = spark.createDataFrame(
    data,
    ["emp_id", "name", "salary"]
)

df.show()
```

---

## Is `show()` a Transformation or Action?

`show()` is an **Action**.

Spark DataFrames follow lazy evaluation.

For example:

```python
filtered_df = df.filter(
    df.salary > 80000
)
```

Spark does not immediately process the data.

Execution starts when we call:

```python
filtered_df.show()
```

Flow:

```text
DataFrame
   ↓
Transformation
   ↓
Logical Plan
   ↓
show()
   ↓
ACTION
   ↓
Spark Job
   ↓
Stages
   ↓
Tasks
   ↓
Executors
   ↓
Rows returned
```

---

## `show()` vs `collect()`

### `show()`

```text
Purpose:
View records

Returns:
None

Driver:
Only records needed for display are returned

Use:
Debugging / inspection
```

### `collect()`

```text
Purpose:
Return records to Python program

Returns:
Python list of Row objects

Driver:
All selected records are brought to Driver

Risk:
Driver memory issue for huge datasets
```

Example:

```python
rows = df.collect()

print(rows)
```

---

## Interview Answer

`show()` is a Spark action used to display DataFrame rows for inspection and debugging. It triggers Spark execution and displays 20 rows by default.

---

# 3.2 `printSchema()`

## Definition

`printSchema()` is used to display the structure of a DataFrame.

It tells us:

- Column names
- Data types
- Nullability
- Nested structures

---

## Syntax

```python
df.printSchema()
```

Example output:

```text
root
 |-- customer_id: string (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: long (nullable = true)
```

---

## Understanding the Output

Take:

```text
amount: long (nullable = true)
```

Meaning:

```text
amount
→ Column name

long
→ Spark datatype

nullable = true
→ Column can contain NULL
```

---

## Why `printSchema()` Is Important

Suppose we read a CSV:

```python
df = spark.read \
    .option("header", True) \
    .csv("sales.csv")
```

Then:

```python
df.printSchema()
```

We may see:

```text
amount: string
quantity: string
```

But logically they should be numeric.

So `printSchema()` helps detect schema problems before performing transformations.

---

## Common Uses

```text
After reading source data
After casting columns
After joins
After schema evolution
When debugging datatype issues
When working with JSON
```

---

## `show()` vs `printSchema()`

```text
show()
→ Shows actual records

printSchema()
→ Shows structure of records
```

---

## Does `printSchema()` Trigger a Spark Job?

Normally no.

The schema is already known by Spark.

Spark does not need to scan all rows just to print schema metadata.

---

# 3.3 `schema`

## Definition

`schema` returns the complete DataFrame schema as a Spark object.

The returned object is:

```text
StructType
```

---

## Syntax

```python
df.schema
```

Example:

```text
StructType([
    StructField('customer_id', StringType(), True),
    StructField('product', StringType(), True),
    StructField('amount', LongType(), True)
])
```

---

# StructType

`StructType` represents the entire schema.

Conceptually:

```text
StructType
│
├── StructField
├── StructField
└── StructField
```

---

# StructField

Each column is represented by a `StructField`.

Example:

```text
StructField(
    'amount',
    LongType(),
    True
)
```

Meaning:

```text
amount
→ Column name

LongType()
→ Datatype

True
→ Nullable
```

---

## Access All Fields

```python
df.schema.fields
```

---

## Access One Column

```python
df.schema["amount"]
```

---

## Access Column Name

```python
df.schema["amount"].name
```

---

## Access Datatype

```python
df.schema["amount"].dataType
```

---

## Access Nullability

```python
df.schema["amount"].nullable
```

---

## Loop Through Schema

```python
for field in df.schema.fields:
    print(
        field.name,
        field.dataType,
        field.nullable
    )
```

---

## Real Project Use

We can dynamically identify string columns.

```python
from pyspark.sql.types import StringType

string_columns = [
    field.name
    for field in df.schema.fields
    if isinstance(
        field.dataType,
        StringType
    )
]
```

This is useful when building reusable ETL frameworks.

---

## `schema` vs `printSchema()`

```text
printSchema()
→ Prints schema
→ Human-readable

schema
→ Returns StructType object
→ Programmatically usable
```

---

## Important Syntax

Correct:

```python
df.schema
```

```python
df.schema["amount"]
```

Incorrect:

```python
df.schema()
```

because `schema` is a property.

---

# 3.4 `columns`

## Definition

`columns` returns all DataFrame column names as a Python list.

---

## Syntax

```python
df.columns
```

Example output:

```python
[
    "customer_id",
    "product",
    "amount"
]
```

---

## Important

`columns` is a property.

Correct:

```python
df.columns
```

Incorrect:

```python
df.columns()
```

---

## Number of Columns

```python
len(df.columns)
```

---

## First Column

```python
df.columns[0]
```

---

## Last Column

```python
df.columns[-1]
```

---

## Loop Through Columns

```python
for column_name in df.columns:
    print(column_name)
```

---

## Check Whether Column Exists

```python
if "amount" in df.columns:
    print("Column exists")
```

---

## Check Missing Column

```python
if "salary" not in df.columns:
    print("Column missing")
```

---

## Real Project Use

Suppose target requires:

```python
required_columns = [
    "customer_id",
    "product",
    "amount"
]
```

Find missing columns:

```python
missing_columns = [
    c
    for c in required_columns
    if c not in df.columns
]
```

This is useful for ingestion validation.

---

# 3.5 `dtypes`

## Definition

`dtypes` returns DataFrame column names together with their datatypes.

---

## Syntax

```python
df.dtypes
```

Example output:

```python
[
    ('customer_id', 'string'),
    ('product', 'string'),
    ('amount', 'bigint')
]
```

---

## Return Type

`dtypes` returns:

```text
Python list
```

Each element is:

```text
tuple
```

Format:

```text
(column_name, datatype)
```

---

## Loop Through Data Types

```python
for name, dtype in df.dtypes:
    print(
        name,
        dtype
    )
```

---

## Find String Columns

```python
string_columns = [
    name
    for name, dtype in df.dtypes
    if dtype == "string"
]
```

---

## Find Numeric Columns

```python
numeric_types = [
    "int",
    "bigint",
    "double",
    "float"
]

numeric_columns = [
    name
    for name, dtype in df.dtypes
    if dtype in numeric_types
]
```

---

## Difference

```text
columns
→ Names only

dtypes
→ Names + datatype

schema
→ Complete schema details
```

---

# 3.6 `count()`

## Definition

`count()` returns the total number of rows in a DataFrame.

---

## Syntax

```python
df.count()
```

Example:

```python
row_count = df.count()

print(row_count)
```

---

## `count()` Is an Action

Unlike `columns`, `schema`, and `dtypes`, `count()` requires Spark to execute the query.

Flow:

```text
DataFrame
   ↓
count()
   ↓
Spark Job
   ↓
Executors process partitions
   ↓
Partial counts
   ↓
Final aggregation
   ↓
Driver receives final number
```

---

## Important

`count()` does not send all rows to the Driver.

Suppose:

```text
Partition 1 → 100
Partition 2 → 200
Partition 3 → 300
```

Spark combines:

```text
100 + 200 + 300
=
600
```

Driver receives:

```text
600
```

not all 600 rows.

---

## `count()` vs `collect()`

```text
count()
→ Returns number of rows

collect()
→ Returns actual rows
```

---

## Row Count vs Column Count

Rows:

```python
df.count()
```

Columns:

```python
len(df.columns)
```

---

## Real ETL Example

```python
source_count = source_df.count()

valid_count = valid_df.count()

reject_count = reject_df.count()
```

Validation:

```python
if source_count == valid_count + reject_count:
    print("Reconciliation successful")
```

---

## Performance

`count()` may be expensive on huge datasets.

Avoid unnecessary repeated counts.

---

# 3.7 `describe()`

## Definition

`describe()` generates basic statistical information for DataFrame columns.

It normally provides:

```text
count
mean
stddev
min
max
```

---

## Syntax

```python
df.describe().show()
```

Specific column:

```python
df.describe(
    "amount"
).show()
```

Multiple columns:

```python
df.describe(
    "amount",
    "quantity"
).show()
```

---

## Example Output

```text
+-------+--------+
|summary|  amount|
+-------+--------+
|count  |       4|
|mean   | 35000.0|
|stddev |12909...|
|min    |   20000|
|max    |   50000|
+-------+--------+
```

---

## Meaning

### count

Number of non-null values.

### mean

Average value.

### stddev

Standard deviation.

It measures how spread out values are around the mean.

```text
Small stddev
→ Values are close together

Large stddev
→ Values are widely spread
```

### min

Minimum value.

### max

Maximum value.

---

## Important Difference

```python
df.count()
```

counts all rows.

But the `count` generated by `describe()` is a per-column non-null count.

---

## `describe()` Returns a DataFrame

Example:

```python
summary_df = df.describe()
```

Then:

```python
summary_df.show()
```

---

## `describe()` vs `summary()`

```text
describe()
→ Basic statistics

summary()
→ More detailed statistics
```

`summary()` may include:

```text
25%
50%
75%
```

---

# 4. COLUMN OPERATIONS

Column operations are used to:

- Select columns
- Create new columns
- Modify existing columns
- Rename columns
- Drop columns
- Convert datatypes
- Create expressions

The most important functions are:

```text
select()
col()
alias()
withColumn()
withColumnRenamed()
drop()
cast()
```

---

# 4.1 `select()`

## Definition

`select()` is used to choose columns or expressions from a DataFrame.

---

## Syntax

```python
df.select(
    "customer_id",
    "amount"
)
```

---

## Select One Column

```python
df.select(
    "customer_id"
)
```

---

## Select Multiple Columns

```python
df.select(
    "customer_id",
    "product",
    "amount"
)
```

---

## Select Using `col()`

```python
from pyspark.sql.functions import col

df.select(
    col("customer_id"),
    col("amount")
)
```

---

## Select All Columns

```python
df.select("*")
```

---

## Dynamic Selection

```python
required_columns = [
    "customer_id",
    "product",
    "amount"
]

df.select(
    *required_columns
)
```

The `*` expands the Python list.

---

## Create Expression With `select()`

```python
df.select(
    "customer_id",
    (
        col("amount") * 2
    ).alias(
        "double_amount"
    )
)
```

---

## Select All Columns Plus New Column

```python
df.select(
    "*",
    (
        col("amount") * 0.18
    ).alias(
        "tax"
    )
)
```

---

## Is `select()` an Action?

No.

It is a Transformation.

Spark uses lazy evaluation.

```text
select()
→ Build logical plan

show()
→ Execute logical plan
```

---

## Projection Pruning

Suppose a Parquet file contains 100 columns.

But we need only:

```python
df.select(
    "customer_id",
    "amount"
)
```

Spark's Catalyst optimizer may use:

```text
Projection Pruning
```

Meaning unnecessary columns may not be read.

Benefits:

- Less disk I/O
- Less memory
- Less data transfer
- Faster execution

---

# 4.2 `col()`

## Definition

`col()` creates a Spark Column object.

---

## Import

```python
from pyspark.sql.functions import col
```

---

## Syntax

```python
col("amount")
```

---

## Difference

```text
"amount"
→ Python string

col("amount")
→ Spark Column object
```

A Column object can be used in expressions.

---

## Arithmetic

```python
col("amount") * 2
```

---

## Comparison

```python
col("amount") > 50000
```

---

## Filter

```python
df.filter(
    col("amount") > 50000
)
```

---

## Multiple Conditions

AND:

```python
df.filter(
    (col("amount") > 20000)
    &
    (col("product") == "Laptop")
)
```

OR:

```python
df.filter(
    (col("amount") > 50000)
    |
    (col("product") == "Mobile")
)
```

NOT:

```python
df.filter(
    ~(col("product") == "Mobile")
)
```

Operators:

```text
& → AND
| → OR
~ → NOT
```

---

## Null Checks

```python
col("amount").isNull()
```

```python
col("amount").isNotNull()
```

---

## Cast

```python
col("amount").cast(
    "double"
)
```

---

## Alias

```python
col("amount").alias(
    "sales_amount"
)
```

---

# 4.3 `alias()`

## Definition

`alias()` assigns a temporary name to:

- Column
- Expression
- DataFrame

---

## Column Alias

```python
df.select(
    col("amount").alias(
        "sales_amount"
    )
)
```

---

## Expression Alias

```python
df.select(
    (
        col("amount") * 2
    ).alias(
        "double_amount"
    )
)
```

---

## Aggregation Alias

```python
from pyspark.sql.functions import sum

df.groupBy(
    "product"
).agg(
    sum(
        "amount"
    ).alias(
        "total_amount"
    )
)
```

---

## DataFrame Alias

Useful in joins.

```python
customer = customer_df.alias("c")

order = order_df.alias("o")
```

Join:

```python
joined_df = customer.join(
    order,
    col("c.customer_id")
    ==
    col("o.customer_id")
)
```

---

## Why DataFrame Alias Is Important

Suppose both DataFrames contain:

```text
customer_id
```

Without alias, column references can become ambiguous.

Alias allows:

```text
c.customer_id
o.customer_id
```

---

## `alias()` vs `withColumnRenamed()`

```text
alias()
→ Temporary naming
→ Used in expressions, aggregates and joins

withColumnRenamed()
→ Renames existing DataFrame column
```

---

# 4.4 `withColumn()`

## Definition

`withColumn()` is used to:

```text
Add a new column

OR

Replace an existing column
```

---

## Syntax

```python
df.withColumn(
    "column_name",
    expression
)
```

---

## Add New Column

```python
df = df.withColumn(
    "tax",
    col("amount") * 0.18
)
```

---

## Modify Existing Column

```python
df = df.withColumn(
    "amount",
    col("amount") * 2
)
```

Rule:

```text
New column name
→ Column added

Existing column name
→ Column replaced
```

---

## Constant Column Using `lit()`

```python
from pyspark.sql.functions import lit

df = df.withColumn(
    "country",
    lit("India")
)
```

---

## Conditional Column Using `when()`

```python
from pyspark.sql.functions import when

df = df.withColumn(
    "category",

    when(
        col("amount") >= 50000,
        "HIGH"
    )

    .otherwise(
        "NORMAL"
    )
)
```

---

## Multiple Conditions

```python
df = df.withColumn(
    "category",

    when(
        col("amount") >= 50000,
        "HIGH"
    )

    .when(
        col("amount") >= 30000,
        "MEDIUM"
    )

    .otherwise(
        "LOW"
    )
)
```

---

## Null Handling

```python
from pyspark.sql.functions import coalesce, lit

df = df.withColumn(
    "amount",

    coalesce(
        col("amount"),
        lit(0)
    )
)
```

---

## String Cleaning

```python
from pyspark.sql.functions import trim, upper

df = df.withColumn(
    "customer_name",

    upper(
        trim(
            col("customer_name")
        )
    )
)
```

---

## Multiple `withColumn()` Calls

```python
df = (
    df

    .withColumn(
        "tax",
        col("amount") * 0.18
    )

    .withColumn(
        "final_amount",
        col("amount") * 1.18
    )

    .withColumn(
        "country",
        lit("India")
    )
)
```

---

## DataFrame Immutability

Spark DataFrames are immutable.

```python
new_df = df.withColumn(
    "tax",
    col("amount") * 0.18
)
```

This does not modify the original `df`.

It creates a new DataFrame.

---

## Performance Note

A few `withColumn()` calls are normal.

But repeatedly adding a very large number of columns can create a large logical plan.

For many derived columns, sometimes this is cleaner:

```python
df.select(
    "*",
    expression1.alias("col1"),
    expression2.alias("col2"),
    expression3.alias("col3")
)
```

---

# 4.5 `withColumnRenamed()`

## Definition

`withColumnRenamed()` changes the name of an existing DataFrame column.

---

## Syntax

```python
df.withColumnRenamed(
    "old_name",
    "new_name"
)
```

---

## Example

```python
df = df.withColumnRenamed(
    "customer_name",
    "name"
)
```

---

## Rename Multiple Columns

```python
df = (
    df

    .withColumnRenamed(
        "customer_id",
        "cust_id"
    )

    .withColumnRenamed(
        "customer_name",
        "name"
    )

    .withColumnRenamed(
        "amount",
        "sales_amount"
    )
)
```

---

## Dynamic Renaming

```python
rename_map = {
    "customer_id": "cust_id",
    "customer_name": "name",
    "amount": "sales_amount"
}

for old_name, new_name in rename_map.items():

    df = df.withColumnRenamed(
        old_name,
        new_name
    )
```

---

## Standardize Column Names

Suppose:

```text
Customer ID
Customer Name
Order Amount
```

We want:

```text
customer_id
customer_name
order_amount
```

Code:

```python
for column_name in df.columns:

    new_name = (
        column_name
        .lower()
        .replace(" ", "_")
    )

    df = df.withColumnRenamed(
        column_name,
        new_name
    )
```

---

## Important Difference

```text
withColumn()
→ Change column value or create column

withColumnRenamed()
→ Change column name only
```

---

# 4.6 `drop()`

## Definition

`drop()` removes one or more columns from a DataFrame.

---

## Syntax

```python
df.drop(
    "city"
)
```

---

## Drop Multiple Columns

```python
df.drop(
    "city",
    "product"
)
```

---

## Dynamic Drop

```python
columns_to_drop = [
    "city",
    "product"
]

df = df.drop(
    *columns_to_drop
)
```

---

## Real Project Example

Raw ingestion may add:

```text
batch_id
source_file
ingestion_timestamp
record_hash
```

Remove them:

```python
df = df.drop(
    "batch_id",
    "source_file",
    "ingestion_timestamp",
    "record_hash"
)
```

---

## Drop After Join

Suppose:

```python
joined_df = customer_df.join(
    order_df,
    customer_df.customer_id
    ==
    order_df.customer_id
)
```

Duplicate join keys may exist.

Drop one:

```python
joined_df = joined_df.drop(
    order_df.customer_id
)
```

---

## `drop()` vs `select()`

Use `drop()` when:

```text
You need most columns
but want to remove a few
```

Use `select()` when:

```text
You need only a few specific columns
```

---

## `drop()` vs `filter()`

```text
drop()
→ Removes columns

filter()
→ Removes rows
```

---

# 4.7 `cast()`

## Definition

`cast()` converts a column from one datatype to another.

---

## Syntax

```python
col(
    "column_name"
).cast(
    "datatype"
)
```

---

## Common Datatypes

```text
string
int
integer
long
bigint
float
double
boolean
date
timestamp
decimal(p,s)
```

---

## String to Integer

```python
df = df.withColumn(
    "age",
    col("age").cast(
        "int"
    )
)
```

---

## String to Double

```python
df = df.withColumn(
    "amount",
    col("amount").cast(
        "double"
    )
)
```

---

## String to Long

```python
df = df.withColumn(
    "customer_id",
    col("customer_id").cast(
        "long"
    )
)
```

---

## String to Boolean

```python
df = df.withColumn(
    "is_active",
    col("is_active").cast(
        "boolean"
    )
)
```

---

## String to Date

```python
df = df.withColumn(
    "order_date",
    col("order_date").cast(
        "date"
    )
)
```

---

## String to Timestamp

```python
df = df.withColumn(
    "event_time",
    col("event_time").cast(
        "timestamp"
    )
)
```

---

## Decimal Datatype

For financial data:

```python
df = df.withColumn(
    "amount",

    col(
        "amount"
    ).cast(
        "decimal(18,2)"
    )
)
```

Meaning:

```text
18
→ Total digits

2
→ Digits after decimal
```

Example:

```text
123456.78
```

---

## Why Decimal Is Important

`double` uses floating-point representation.

For financial calculations where exact decimal precision is important, `decimal(p,s)` is usually preferred.

---

## Custom Date Format

Suppose data is:

```text
23/09/2026
```

Use:

```python
from pyspark.sql.functions import to_date

df = df.withColumn(
    "order_date",

    to_date(
        col("order_date"),
        "dd/MM/yyyy"
    )
)
```

---

## Custom Timestamp Format

```python
from pyspark.sql.functions import to_timestamp

df = df.withColumn(
    "event_time",

    to_timestamp(
        col("event_time"),
        "dd-MM-yyyy HH:mm:ss"
    )
)
```

---

## Invalid Cast

Suppose:

```text
amount

50000
30000
ABC
20000
```

Casting:

```python
col("amount").cast(
    "double"
)
```

Depending on Spark ANSI configuration:

```text
ABC
→ May become NULL

OR

→ May produce a cast error
```

---

## Detect Failed Cast

Keep original and converted columns:

```python
df2 = df.withColumn(
    "amount_casted",

    col(
        "amount"
    ).cast(
        "double"
    )
)
```

Find invalid records:

```python
invalid_df = df2.filter(

    col(
        "amount"
    ).isNotNull()

    &

    col(
        "amount_casted"
    ).isNull()
)
```

This means:

```text
Original value exists
+
Converted value is NULL
=
Possible invalid datatype
```

---

# COMPLETE PRACTICAL EXAMPLE

## Create Spark Session

```python
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName(
        "DataFramePractice"
    )
    .getOrCreate()
)
```

---

## Import Functions

```python
from pyspark.sql.functions import (
    col,
    lit,
    when,
    trim,
    upper
)
```

---

## Create Sample Data

```python
data = [
    (
        "101",
        "  Anuj  ",
        "Laptop",
        "50000.50",
        "Ranchi"
    ),
    (
        "102",
        "Rahul",
        "Mobile",
        "20000.75",
        "Delhi"
    ),
    (
        "103",
        "Neha",
        "Laptop",
        "35000.00",
        "Pune"
    ),
    (
        "104",
        "Priya",
        "TV",
        "60000.00",
        "Delhi"
    )
]
```

---

## Create DataFrame

```python
columns = [
    "customer_id",
    "customer_name",
    "product",
    "amount",
    "city"
]

df = spark.createDataFrame(
    data,
    columns
)
```

---

## Inspect Data

```python
df.show(
    truncate=False
)
```

---

## Inspect Schema

```python
df.printSchema()
```

---

## Cast Datatypes

```python
df = (
    df

    .withColumn(
        "customer_id",
        col("customer_id").cast("int")
    )

    .withColumn(
        "amount",
        col("amount").cast(
            "decimal(12,2)"
        )
    )
)
```

---

## Clean Name

```python
df = df.withColumn(
    "customer_name",

    upper(
        trim(
            col("customer_name")
        )
    )
)
```

---

## Add Tax

```python
df = df.withColumn(
    "tax",
    col("amount") * 0.18
)
```

---

## Add Final Amount

```python
df = df.withColumn(
    "final_amount",
    col("amount") * 1.18
)
```

---

## Create Category

```python
df = df.withColumn(
    "amount_category",

    when(
        col("amount") >= 50000,
        "HIGH"
    )

    .when(
        col("amount") >= 30000,
        "MEDIUM"
    )

    .otherwise(
        "LOW"
    )
)
```

---

## Add Country

```python
df = df.withColumn(
    "country",
    lit("India")
)
```

---

## Rename Column

```python
df = df.withColumnRenamed(
    "customer_id",
    "cust_id"
)
```

---

## Drop City

```python
df = df.drop(
    "city"
)
```

---

## Final Select

```python
final_df = df.select(
    "cust_id",
    "customer_name",
    "product",
    "amount",
    "tax",
    "final_amount",
    "amount_category",
    "country"
)
```

---

## Final Output

```python
final_df.show(
    truncate=False
)
```

---

# TRANSFORMATION VS ACTION

## Transformations

Transformations create a new DataFrame and are lazily evaluated.

Examples:

```text
select()
filter()
withColumn()
withColumnRenamed()
drop()
```

They do not immediately execute the complete Spark job.

---

## Actions

Actions trigger execution.

Examples:

```text
show()
count()
collect()
```

---

# EXECUTION FLOW

```text
PySpark DataFrame Code
        |
        v

Transformations
        |
        v

Unresolved Logical Plan
        |
        v

Analyzed Logical Plan
        |
        v

Optimized Logical Plan
        |
        v

Physical Plan
        |
        v

ACTION
        |
        v

DAG
        |
        v

Stages
        |
        v

Tasks
        |
        v

Executors
```

---

# QUICK INTERVIEW REVISION

## `show()` vs `collect()`

```text
show()
→ Display records

collect()
→ Return records to Driver
```

---

## `printSchema()` vs `schema`

```text
printSchema()
→ Human-readable schema

schema
→ StructType object
```

---

## `columns` vs `dtypes`

```text
columns
→ Names

dtypes
→ Names + types
```

---

## `select()` vs `withColumn()`

```text
select()
→ Choose/build output columns

withColumn()
→ Add or modify column
```

---

## `alias()` vs `withColumnRenamed()`

```text
alias()
→ Temporary name

withColumnRenamed()
→ Rename existing column
```

---

## `drop()` vs `filter()`

```text
drop()
→ Remove columns

filter()
→ Remove rows
```

---

## `cast()` vs `withColumn()`

```text
cast()
→ Datatype conversion expression

withColumn()
→ Add/replace DataFrame column
```

---

# COMPLETE MEMORY MAP

```text
DATAFRAME UNDERSTANDING
│
├── show()
│   └── Display records
│
├── printSchema()
│   └── Display structure
│
├── schema
│   └── Access complete schema
│
├── columns
│   └── Get column names
│
├── dtypes
│   └── Get names + datatypes
│
├── count()
│   └── Count rows
│
└── describe()
    └── Statistical summary


COLUMN OPERATIONS
│
├── select()
│   └── Select columns
│
├── col()
│   └── Reference column
│
├── alias()
│   └── Temporary rename
│
├── withColumn()
│   ├── Add column
│   └── Modify column
│
├── withColumnRenamed()
│   └── Rename column
│
├── drop()
│   └── Remove column
│
└── cast()
    └── Convert datatype
```

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("DataFrameLearning")
    .master("local[*]")
    .getOrCreate()
)

# data = [
#     ("C101", "Laptop", 65000),
#     ("C102", "Mobile", 30000),
#     ("C103", "Keyboard", 2000)
# ]

# columns = ["customer_id", "product", "amount"]

# df = spark.createDataFrame(data, columns)

# df.show()

# 5. FILTERING IN PYSPARK

Filtering means selecting only those rows from a DataFrame that satisfy a condition.

For example:

```text
Original Data

+-----------+-------+------+------+
|customer_id|product|amount|city  |
+-----------+-------+------+------+
|101        |Laptop |50000 |Ranchi|
|102        |Mobile |20000 |Delhi |
|103        |Laptop |35000 |Pune  |
|104        |TV     |60000 |Delhi |
|105        |Mobile |25000 |Ranchi|
+-----------+-------+------+------+
```

If we want only records where:

```text
amount > 30000
```

then filtering returns:

```text
+-----------+-------+------+------+
|customer_id|product|amount|city  |
+-----------+-------+------+------+
|101        |Laptop |50000 |Ranchi|
|103        |Laptop |35000 |Pune  |
|104        |TV     |60000 |Delhi |
+-----------+-------+------+------+
```

Filtering is one of the most frequently used operations in data engineering.

Common filtering operations include:

```text
filter()
where()
AND / OR / NOT
isin()
between()
NULL handling
```

---

# Sample DataFrame for Practice

```python
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = (
    SparkSession
    .builder
    .appName("FilteringPractice")
    .getOrCreate()
)
```

Create sample data:

```python
data = [
    (101, "Anuj", "Laptop", 50000, "Ranchi"),
    (102, "Rahul", "Mobile", 20000, "Delhi"),
    (103, "Neha", "Laptop", 35000, "Pune"),
    (104, "Priya", "TV", 60000, "Delhi"),
    (105, "Amit", "Mobile", 25000, "Ranchi"),
    (106, "Riya", None, 45000, None),
    (107, "Karan", "Laptop", None, "Delhi")
]

columns = [
    "customer_id",
    "customer_name",
    "product",
    "amount",
    "city"
]

df = spark.createDataFrame(
    data,
    columns
)

df.show()
```

---

# 5.1 `filter()`

## Definition

`filter()` is used to return only those rows that satisfy a condition.

It removes rows that do not satisfy the condition.

---

## Syntax

```python
df.filter(
    condition
)
```

Example:

```python
df.filter(
    col("amount") > 30000
)
```

To see the output:

```python
df.filter(
    col("amount") > 30000
).show()
```

---

## Example

```python
df.filter(
    col("amount") > 30000
).show()
```

Output conceptually:

```text
+-----------+-------------+-------+------+------+
|customer_id|customer_name|product|amount|city  |
+-----------+-------------+-------+------+------+
|101        |Anuj         |Laptop |50000 |Ranchi|
|103        |Neha         |Laptop |35000 |Pune  |
|104        |Priya        |TV     |60000 |Delhi |
|106        |Riya         |NULL   |45000 |NULL  |
+-----------+-------------+-------+------+------+
```

---

## `filter()` Is a Transformation

`filter()` does not immediately execute the Spark job.

Example:

```python
filtered_df = df.filter(
    col("amount") > 30000
)
```

At this point Spark creates a new logical plan.

Execution happens when we call an action:

```python
filtered_df.show()
```

Flow:

```text
DataFrame
   ↓
filter()
   ↓
Transformation
   ↓
Logical Plan
   ↓
show()
   ↓
Action
   ↓
Spark Execution
```

---

## Common Comparison Operators

```text
==    Equal to
!=    Not equal to
>     Greater than
<     Less than
>=    Greater than or equal to
<=    Less than or equal to
```

---

## Equal Condition

```python
df.filter(
    col("city") == "Delhi"
).show()
```

---

## Not Equal

```python
df.filter(
    col("city") != "Delhi"
).show()
```

---

## Greater Than

```python
df.filter(
    col("amount") > 30000
).show()
```

---

## Greater Than or Equal

```python
df.filter(
    col("amount") >= 30000
).show()
```

---

## Less Than

```python
df.filter(
    col("amount") < 30000
).show()
```

---

## Less Than or Equal

```python
df.filter(
    col("amount") <= 30000
).show()
```

---

# String Style in `filter()`

We can also write conditions as SQL-style strings.

Example:

```python
df.filter(
    "amount > 30000"
).show()
```

This also works.

Another example:

```python
df.filter(
    "city = 'Delhi'"
).show()
```

---

## Column Expression vs SQL String

Both are valid:

```python
df.filter(
    col("amount") > 30000
)
```

and:

```python
df.filter(
    "amount > 30000"
)
```

For larger PySpark code, using `col()` is often easier to combine with functions and dynamic logic.

---

# 5.2 `where()`

## Definition

`where()` is another function used to filter rows.

In PySpark:

```text
where()
and
filter()
```

are effectively aliases for the same operation.

---

## Syntax

```python
df.where(
    condition
)
```

Example:

```python
df.where(
    col("amount") > 30000
).show()
```

---

## Equivalent Code

These two are equivalent:

```python
df.filter(
    col("amount") > 30000
)
```

and:

```python
df.where(
    col("amount") > 30000
)
```

---

## Why Does Spark Have Both?

`where()` is familiar to SQL developers because SQL uses:

```sql
WHERE amount > 30000
```

So PySpark provides both styles.

---

## Example

```python
df.where(
    col("city") == "Delhi"
).show()
```

Same as:

```python
df.filter(
    col("city") == "Delhi"
).show()
```

---

## `filter()` vs `where()`

```text
filter()
→ Common DataFrame API naming

where()
→ SQL-style naming

Functionally:
→ Same purpose
→ Same result
```

---

## Interview Answer

`filter()` and `where()` are equivalent in PySpark. Both return rows that satisfy a condition.

---

# 5.3 AND / OR / NOT

Many real business conditions require more than one rule.

For example:

```text
amount > 30000
AND
city = Delhi
```

or:

```text
product = Laptop
OR
product = Mobile
```

PySpark uses:

```text
& → AND
| → OR
~ → NOT
```

---

# AND Condition

Use:

```text
&
```

Example:

```python
df.filter(
    (col("amount") > 30000)
    &
    (col("city") == "Delhi")
).show()
```

Meaning:

```text
amount > 30000
AND
city = Delhi
```

Both conditions must be true.

---

## Example

Data:

```text
101 Laptop 50000 Ranchi
103 Laptop 35000 Pune
104 TV     60000 Delhi
```

Condition:

```text
amount > 30000
AND
city = Delhi
```

Only:

```text
104 TV 60000 Delhi
```

matches.

---

# OR Condition

Use:

```text
|
```

Example:

```python
df.filter(
    (col("city") == "Delhi")
    |
    (col("city") == "Ranchi")
).show()
```

Meaning:

```text
city = Delhi
OR
city = Ranchi
```

At least one condition must be true.

---

# NOT Condition

Use:

```text
~
```

Example:

```python
df.filter(
    ~(col("city") == "Delhi")
).show()
```

Meaning:

```text
NOT city = Delhi
```

---

## Another NOT Example

```python
df.filter(
    ~col("product").isin(
        "Laptop",
        "Mobile"
    )
).show()
```

Meaning:

```text
product is NOT Laptop
AND
product is NOT Mobile
```

---

# Very Important: Use Parentheses

Correct:

```python
df.filter(
    (col("amount") > 30000)
    &
    (col("city") == "Delhi")
)
```

Avoid writing:

```python
df.filter(
    col("amount") > 30000
    &
    col("city") == "Delhi"
)
```

Because Python operator precedence can create incorrect expressions.

Recommended rule:

```text
Always put each condition inside parentheses.
```

---

# Complex Condition Example

Requirement:

```text
amount >= 30000

AND

city is Delhi OR Ranchi
```

Code:

```python
df.filter(
    (col("amount") >= 30000)
    &
    (
        (col("city") == "Delhi")
        |
        (col("city") == "Ranchi")
    )
).show()
```

---

# Another Complex Example

Requirement:

```text
product = Laptop

AND

amount > 30000

AND

city != Pune
```

Code:

```python
df.filter(
    (col("product") == "Laptop")
    &
    (col("amount") > 30000)
    &
    (col("city") != "Pune")
).show()
```

---

# SQL String Version

AND:

```python
df.filter(
    "amount > 30000 AND city = 'Delhi'"
).show()
```

OR:

```python
df.filter(
    "city = 'Delhi' OR city = 'Ranchi'"
).show()
```

NOT:

```python
df.filter(
    "NOT city = 'Delhi'"
).show()
```

---

# 5.4 `isin()`

## Definition

`isin()` checks whether a column value exists in a given list of values.

It is similar to SQL:

```sql
IN (...)
```

---

## Syntax

```python
col("column_name").isin(
    value1,
    value2,
    value3
)
```

---

## Example

Suppose we want:

```text
product = Laptop
OR
product = Mobile
```

Instead of:

```python
df.filter(
    (col("product") == "Laptop")
    |
    (col("product") == "Mobile")
)
```

we can write:

```python
df.filter(
    col("product").isin(
        "Laptop",
        "Mobile"
    )
).show()
```

This is cleaner.

---

## SQL Equivalent

PySpark:

```python
col("product").isin(
    "Laptop",
    "Mobile"
)
```

SQL:

```sql
product IN ('Laptop', 'Mobile')
```

---

## Using Python List

```python
products = [
    "Laptop",
    "Mobile"
]
```

Then:

```python
df.filter(
    col("product").isin(
        products
    )
).show()
```

You may also commonly see:

```python
df.filter(
    col("product").isin(
        *products
    )
).show()
```

The exact accepted form can depend on how the values are supplied, but passing the list directly is commonly supported in modern PySpark.

---

## NOT IN

Use `~`.

Example:

```python
df.filter(
    ~col("product").isin(
        "Laptop",
        "Mobile"
    )
).show()
```

SQL equivalent:

```sql
product NOT IN ('Laptop', 'Mobile')
```

---

## Real Project Example

Suppose valid transaction statuses are:

```text
SUCCESS
PENDING
PROCESSING
```

Code:

```python
valid_statuses = [
    "SUCCESS",
    "PENDING",
    "PROCESSING"
]

valid_df = df.filter(
    col("status").isin(
        valid_statuses
    )
)
```

---

# 5.5 `between()`

## Definition

`between()` checks whether a value lies within a range.

The boundaries are inclusive.

That means:

```text
between(100, 500)
```

means:

```text
>= 100
AND
<= 500
```

---

## Syntax

```python
col("column_name").between(
    lower_value,
    upper_value
)
```

---

## Example

```python
df.filter(
    col("amount").between(
        20000,
        40000
    )
).show()
```

Equivalent to:

```python
df.filter(
    (col("amount") >= 20000)
    &
    (col("amount") <= 40000)
).show()
```

---

## Important

`between()` is inclusive.

Example:

```python
col("amount").between(
    20000,
    40000
)
```

will include:

```text
20000
25000
30000
35000
40000
```

---

# Date Range With `between()`

Suppose `order_date` is a proper date column.

```python
df.filter(
    col("order_date").between(
        "2026-09-01",
        "2026-09-30"
    )
).show()
```

This can be used for date range filtering when the column type and literals are compatible.

For important date logic, explicitly converting values to date types is often safer.

---

## Real Project Example

Requirement:

```text
Select transactions between 10,000 and 50,000
```

Code:

```python
df.filter(
    col("amount").between(
        10000,
        50000
    )
)
```

---

# NOT BETWEEN

Use `~`.

```python
df.filter(
    ~col("amount").between(
        20000,
        40000
    )
).show()
```

Meaning:

```text
amount < 20000

OR

amount > 40000
```

Note that NULL behavior follows Spark SQL three-valued logic, so NULL values will not automatically behave like ordinary numbers.

---

# 5.6 NULL HANDLING

NULL represents a missing or unknown value.

Example:

```text
+-----------+-------+------+------+
|customer_id|product|amount|city  |
+-----------+-------+------+------+
|101        |Laptop |50000 |Ranchi|
|106        |NULL   |45000 |NULL  |
|107        |Laptop |NULL  |Delhi |
+-----------+-------+------+------+
```

NULL requires special handling.

---

# Why Normal Equality Does Not Work Properly for NULL

Avoid:

```python
df.filter(
    col("city") == None
)
```

Preferred:

```python
df.filter(
    col("city").isNull()
)
```

Similarly, instead of:

```python
col("city") != None
```

use:

```python
col("city").isNotNull()
```

---

# `isNull()`

Used to find NULL values.

Syntax:

```python
col("column_name").isNull()
```

Example:

```python
df.filter(
    col("city").isNull()
).show()
```

This returns records where city is NULL.

---

# `isNotNull()`

Used to find non-NULL values.

Syntax:

```python
col("column_name").isNotNull()
```

Example:

```python
df.filter(
    col("amount").isNotNull()
).show()
```

---

# Multiple NULL Conditions

Find rows where either city or amount is NULL:

```python
df.filter(
    col("city").isNull()
    |
    col("amount").isNull()
).show()
```

---

# Find Rows Where Both Are NULL

```python
df.filter(
    col("city").isNull()
    &
    col("amount").isNull()
).show()
```

---

# Valid Record Example

Suppose a valid record requires:

```text
customer_id IS NOT NULL

AND

amount IS NOT NULL
```

Code:

```python
valid_df = df.filter(
    col("customer_id").isNotNull()
    &
    col("amount").isNotNull()
)
```

---

# Invalid Record Example

```python
invalid_df = df.filter(
    col("customer_id").isNull()
    |
    col("amount").isNull()
)
```

This is very common in ETL validation.

---

# NULL and Comparison Conditions

Suppose:

```text
amount
------
50000
30000
NULL
```

Condition:

```python
df.filter(
    col("amount") > 30000
)
```

NULL does not satisfy the condition.

Why?

Because:

```text
NULL > 30000
```

does not evaluate to TRUE.

It evaluates to UNKNOWN.

Spark filters keep rows only where the condition evaluates to TRUE.

---

# Three-Valued Logic

SQL and Spark use three-valued logic:

```text
TRUE
FALSE
UNKNOWN
```

NULL comparisons often result in:

```text
UNKNOWN
```

Example:

```text
NULL = 10
→ UNKNOWN

NULL > 10
→ UNKNOWN

NULL < 10
→ UNKNOWN
```

Therefore NULL rows are usually removed from normal comparison filters unless NULL is handled explicitly.

---

# NULL-Safe Equality

Spark also supports null-safe equality.

Using Column API:

```python
col("column1").eqNullSafe(
    col("column2")
)
```

This is equivalent to SQL's null-safe comparison operator:

```text
<=>
```

Example:

```python
df.filter(
    col("city").eqNullSafe(
        col("another_city")
    )
)
```

Difference from normal equality:

```text
NULL == NULL
→ UNKNOWN with normal equality semantics

NULL <=> NULL
→ TRUE with null-safe equality
```

This is useful when comparing nullable columns.

---

# NULL HANDLING WITH `fillna()`

Filtering is not the only way to handle NULL values.

We can replace NULLs.

Example:

```python
df.fillna(
    {
        "city": "UNKNOWN",
        "amount": 0
    }
)
```

Equivalent style:

```python
df.na.fill(
    {
        "city": "UNKNOWN",
        "amount": 0
    }
)
```

---

# DROP NULL RECORDS

Remove rows containing NULLs:

```python
df.dropna()
```

or:

```python
df.na.drop()
```

---

## Drop Rows Where Specific Columns Are NULL

```python
df.dropna(
    subset=[
        "customer_id",
        "amount"
    ]
)
```

---

# FILTERING NULL vs REPLACING NULL

Use filtering when:

```text
You want to remove/select records
based on missing values.
```

Example:

```python
df.filter(
    col("amount").isNotNull()
)
```

Use `fillna()` when:

```text
You want to replace missing values.
```

Example:

```python
df.fillna(
    {
        "amount": 0
    }
)
```

---

# PRACTICAL BUSINESS EXAMPLES

## Example 1: High-Value Transactions

Requirement:

```text
amount >= 50000
```

Code:

```python
high_value_df = df.filter(
    col("amount") >= 50000
)
```

---

## Example 2: High-Value Delhi Transactions

Requirement:

```text
amount >= 50000

AND

city = Delhi
```

Code:

```python
result_df = df.filter(
    (col("amount") >= 50000)
    &
    (col("city") == "Delhi")
)
```

---

## Example 3: Delhi or Ranchi Customers

```python
result_df = df.filter(
    col("city").isin(
        "Delhi",
        "Ranchi"
    )
)
```

---

## Example 4: Medium-Value Transactions

Requirement:

```text
amount between 20000 and 40000
```

Code:

```python
result_df = df.filter(
    col("amount").between(
        20000,
        40000
    )
)
```

---

## Example 5: Missing City Records

```python
invalid_df = df.filter(
    col("city").isNull()
)
```

---

## Example 6: Valid Amount Records

```python
valid_df = df.filter(
    col("amount").isNotNull()
)
```

---

## Example 7: Valid Business Record

Requirement:

```text
customer_id is not NULL

AND

amount is not NULL

AND

product is not NULL
```

Code:

```python
valid_df = df.filter(
    col("customer_id").isNotNull()
    &
    col("amount").isNotNull()
    &
    col("product").isNotNull()
)
```

---

## Example 8: Valid Products and Amount Range

Requirement:

```text
product must be Laptop or Mobile

AND

amount must be between 20000 and 50000
```

Code:

```python
result_df = df.filter(
    col("product").isin(
        "Laptop",
        "Mobile"
    )
    &
    col("amount").between(
        20000,
        50000
    )
)
```

---

# FILTERING AND CATALYST OPTIMIZER

Filtering is important not only logically but also for performance.

Suppose:

```python
df = spark.read.parquet(
    "/data/transactions"
)

result_df = df.filter(
    col("amount") > 50000
)
```

Spark's Catalyst Optimizer may push the filter closer to the data source.

This is called:

```text
Predicate Pushdown
```

Conceptually:

```text
Without Pushdown

Read Large Dataset
      ↓
Send Data to Spark
      ↓
Filter amount > 50000


With Predicate Pushdown

Apply filter closer to source
      ↓
Read fewer records
      ↓
Process less data
```

Benefits:

```text
Less I/O
Less memory
Less network transfer
Faster execution
```

Predicate pushdown support depends on:

- Data source
- File format
- Filter expression
- Spark/data source capabilities

Parquet and ORC commonly support useful pushdown optimizations.

---

# FILTERING IS A TRANSFORMATION

All of these operations are transformations:

```text
filter()
where()
isin()
between()
isNull()
isNotNull()
```

They do not immediately trigger Spark execution.

Example:

```python
result_df = df.filter(
    col("amount") > 30000
)
```

Execution happens when an action is called:

```python
result_df.show()
```

or:

```python
result_df.count()
```

or:

```python
result_df.write...
```

---

# `filter()` vs `where()`

```text
filter()
→ DataFrame API style

where()
→ SQL-style name

Both:
→ Same purpose
→ Filter rows
→ Return new DataFrame
```

---

# `isin()` vs OR Conditions

Without `isin()`:

```python
df.filter(
    (col("product") == "Laptop")
    |
    (col("product") == "Mobile")
    |
    (col("product") == "TV")
)
```

With `isin()`:

```python
df.filter(
    col("product").isin(
        "Laptop",
        "Mobile",
        "TV"
    )
)
```

`isin()` is cleaner when checking against multiple values.

---

# `between()` vs AND

Without `between()`:

```python
df.filter(
    (col("amount") >= 20000)
    &
    (col("amount") <= 40000)
)
```

With `between()`:

```python
df.filter(
    col("amount").between(
        20000,
        40000
    )
)
```

Both are equivalent for ordinary non-null values.

---

# IMPORTANT NULL RULES

Remember:

```text
NULL is not equal to anything using normal equality semantics.

NULL is not even normally compared as TRUE to another NULL.

Use:

isNull()
isNotNull()
eqNullSafe()
```

Examples:

```python
col("city").isNull()
```

```python
col("city").isNotNull()
```

```python
col("city1").eqNullSafe(
    col("city2")
)
```

---

# COMMON MISTAKES

## Mistake 1: Using Python `and`

Wrong:

```python
df.filter(
    (col("amount") > 30000)
    and
    (col("city") == "Delhi")
)
```

Correct:

```python
df.filter(
    (col("amount") > 30000)
    &
    (col("city") == "Delhi")
)
```

---

## Mistake 2: Using Python `or`

Wrong:

```python
df.filter(
    (col("city") == "Delhi")
    or
    (col("city") == "Ranchi")
)
```

Correct:

```python
df.filter(
    (col("city") == "Delhi")
    |
    (col("city") == "Ranchi")
)
```

---

## Mistake 3: Forgetting Parentheses

Avoid:

```python
col("amount") > 30000 & col("city") == "Delhi"
```

Correct:

```python
(
    col("amount") > 30000
)
&
(
    col("city") == "Delhi"
)
```

---

## Mistake 4: Comparing NULL With `== None`

Avoid:

```python
col("city") == None
```

Use:

```python
col("city").isNull()
```

---

## Mistake 5: Using `drop()` to Remove Rows

Wrong idea:

```text
drop()
→ Remove rows
```

Actual behavior:

```text
drop()
→ Remove columns

filter()
→ Filter rows
```

---

# INTERVIEW QUESTIONS

## 1. What is `filter()` in PySpark?

`filter()` is a DataFrame transformation used to return only those rows that satisfy a given condition.

---

## 2. Difference between `filter()` and `where()`?

There is no major functional difference. `where()` is an alias of `filter()` and is useful for developers familiar with SQL syntax.

---

## 3. Which operators are used for multiple conditions?

```text
& → AND
| → OR
~ → NOT
```

---

## 4. Why should conditions be enclosed in parentheses?

Because Python operator precedence can otherwise create incorrect expressions or errors.

---

## 5. What does `isin()` do?

`isin()` checks whether a column value exists in a specified set/list of values.

Example:

```python
col("city").isin(
    "Delhi",
    "Ranchi"
)
```

---

## 6. Is `between()` inclusive?

Yes.

```python
col("amount").between(
    100,
    500
)
```

means:

```text
amount >= 100
AND
amount <= 500
```

---

## 7. How do you check for NULL?

```python
col("column_name").isNull()
```

---

## 8. How do you check for non-NULL?

```python
col("column_name").isNotNull()
```

---

## 9. Why should we not use normal equality for NULL?

Because SQL/Spark uses three-valued logic and normal comparisons with NULL generally evaluate to UNKNOWN rather than TRUE.

---

## 10. Is `filter()` an Action?

No.

It is a Transformation and follows lazy evaluation.

---

# QUICK REVISION TABLE

| Operation | Purpose |
|---|---|
| `filter()` | Filter rows based on condition |
| `where()` | Same as `filter()` |
| `&` | AND |
| `\|` | OR |
| `~` | NOT |
| `isin()` | Check values against a list |
| `between()` | Check inclusive range |
| `isNull()` | Find NULL values |
| `isNotNull()` | Find non-NULL values |
| `eqNullSafe()` | NULL-safe equality comparison |
| `fillna()` | Replace NULL values |
| `dropna()` | Remove rows containing NULL values |

---

# COMPLETE MEMORY MAP

```text
5. FILTERING
│
├── filter()
│   ├── Equality
│   ├── Comparison
│   └── SQL expression
│
├── where()
│   └── Same as filter()
│
├── AND / OR / NOT
│   ├── &  → AND
│   ├── |  → OR
│   └── ~  → NOT
│
├── isin()
│   ├── IN
│   └── NOT IN
│
├── between()
│   ├── Lower bound inclusive
│   └── Upper bound inclusive
│
└── NULL Handling
    ├── isNull()
    ├── isNotNull()
    ├── eqNullSafe()
    ├── fillna()
    └── dropna()
```

---

# FINAL PRACTICAL EXAMPLE

Requirement:

```text
Select customers where:

1. amount is not NULL
2. amount is between 20000 and 50000
3. product is Laptop or Mobile
4. city is Delhi or Ranchi
5. customer_name is not NULL
```

Code:

```python
result_df = df.filter(

    col("amount").isNotNull()

    &

    col("amount").between(
        20000,
        50000
    )

    &

    col("product").isin(
        "Laptop",
        "Mobile"
    )

    &

    col("city").isin(
        "Delhi",
        "Ranchi"
    )

    &

    col("customer_name").isNotNull()
)
```

Display:

```python
result_df.show(
    truncate=False
)
```

Execution flow:

```text
Source Data
    ↓
NULL Validation
    ↓
Amount Range Filter
    ↓
Product Filter
    ↓
City Filter
    ↓
Valid Rows
    ↓
show()
    ↓
Spark Job
```

# 6. CONDITIONAL LOGIC IN PYSPARK

Conditional logic is used when we want to create values based on one or more conditions.

It is similar to:

```text
IF
ELSE IF
ELSE
```

in normal programming.

In PySpark, conditional logic is mainly implemented using:

```text
when()
otherwise()
Multiple when() conditions
```

These are commonly used for:

- Creating flags
- Categorizing data
- Applying business rules
- Data-quality classification
- Deriving status columns
- Creating buckets or ranges
- Handling conditional transformations

---

# Sample DataFrame

```python
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when

spark = (
    SparkSession
    .builder
    .appName("ConditionalLogic")
    .getOrCreate()
)
```

Create sample data:

```python
data = [
    (101, "Anuj", 50000, "SUCCESS"),
    (102, "Rahul", 20000, "FAILED"),
    (103, "Neha", 35000, "SUCCESS"),
    (104, "Priya", 60000, "PENDING"),
    (105, "Amit", 25000, "SUCCESS"),
    (106, "Riya", None, "FAILED")
]

columns = [
    "customer_id",
    "customer_name",
    "amount",
    "status"
]

df = spark.createDataFrame(
    data,
    columns
)

df.show()
```

---

# 6.1 `when()`

## Definition

`when()` is used to apply conditional logic to a column.

It is similar to:

```text
IF condition THEN value
```

---

## Import

```python
from pyspark.sql.functions import when
```

Usually we also use:

```python
from pyspark.sql.functions import col
```

---

## Basic Syntax

```python
when(
    condition,
    value
)
```

Example:

```python
when(
    col("amount") >= 50000,
    "HIGH"
)
```

Meaning:

```text
IF amount >= 50000
THEN "HIGH"
```

---

# Using `when()` With `withColumn()`

Most commonly, `when()` is used inside `withColumn()`.

Example:

```python
df2 = df.withColumn(
    "amount_flag",

    when(
        col("amount") >= 50000,
        "HIGH"
    )
)
```

Now `amount_flag` will contain:

```text
HIGH
```

for rows where:

```text
amount >= 50000
```

For rows where the condition is false, Spark will return:

```text
NULL
```

unless we provide `otherwise()`.

---

# Example Without `otherwise()`

```python
df2 = df.withColumn(
    "amount_flag",

    when(
        col("amount") >= 50000,
        "HIGH"
    )
)

df2.show()
```

Conceptual output:

```text
+-----------+-------------+------+-------+-----------+
|customer_id|customer_name|amount|status |amount_flag|
+-----------+-------------+------+-------+-----------+
|101        |Anuj         |50000 |SUCCESS|HIGH       |
|102        |Rahul        |20000 |FAILED |NULL       |
|103        |Neha         |35000 |SUCCESS|NULL       |
|104        |Priya        |60000 |PENDING|HIGH       |
|105        |Amit         |25000 |SUCCESS|NULL       |
|106        |Riya         |NULL  |FAILED |NULL       |
+-----------+-------------+------+-------+-----------+
```

Important:

```text
when()
without otherwise()
→ unmatched rows become NULL
```

---

# 6.2 `otherwise()`

## Definition

`otherwise()` defines what should happen when none of the previous `when()` conditions are true.

It is equivalent to:

```text
ELSE
```

---

## Syntax

```python
when(
    condition,
    value_if_true
).otherwise(
    value_if_false
)
```

---

## Example

```python
df2 = df.withColumn(
    "amount_flag",

    when(
        col("amount") >= 50000,
        "HIGH"
    ).otherwise(
        "NORMAL"
    )
)
```

Meaning:

```text
IF amount >= 50000
    THEN HIGH
ELSE
    NORMAL
```

---

## Output Conceptually

```text
50000 → HIGH
20000 → NORMAL
35000 → NORMAL
60000 → HIGH
25000 → NORMAL
```

---

# SQL Equivalent

PySpark:

```python
when(
    col("amount") >= 50000,
    "HIGH"
).otherwise(
    "NORMAL"
)
```

SQL equivalent:

```sql
CASE
    WHEN amount >= 50000 THEN 'HIGH'
    ELSE 'NORMAL'
END
```

This is an important comparison for interviews.

---

# `when()` + `otherwise()` Example

Requirement:

```text
If status = SUCCESS
    ACTIVE
Else
    INACTIVE
```

Code:

```python
df2 = df.withColumn(
    "status_flag",

    when(
        col("status") == "SUCCESS",
        "ACTIVE"
    ).otherwise(
        "INACTIVE"
    )
)
```

---

# 6.3 MULTIPLE CONDITIONS

Real business logic usually requires more than one condition.

Example:

```text
amount >= 50000
→ HIGH

amount >= 30000
→ MEDIUM

otherwise
→ LOW
```

This is equivalent to:

```text
IF amount >= 50000
    HIGH

ELSE IF amount >= 30000
    MEDIUM

ELSE
    LOW
```

---

# Syntax for Multiple `when()`

```python
when(
    condition1,
    value1
)

.when(
    condition2,
    value2
)

.when(
    condition3,
    value3
)

.otherwise(
    default_value
)
```

---

# Example

```python
df2 = df.withColumn(
    "amount_category",

    when(
        col("amount") >= 50000,
        "HIGH"
    )

    .when(
        col("amount") >= 30000,
        "MEDIUM"
    )

    .otherwise(
        "LOW"
    )
)
```

---

# How Spark Evaluates Multiple `when()` Conditions

Spark evaluates conditions from top to bottom.

The first condition that evaluates to TRUE wins.

Example:

```python
when(
    col("amount") >= 50000,
    "HIGH"
)

.when(
    col("amount") >= 30000,
    "MEDIUM"
)
```

For:

```text
amount = 60000
```

Both conditions are technically true:

```text
60000 >= 50000 → TRUE

60000 >= 30000 → TRUE
```

But Spark stops at the first matching condition:

```text
HIGH
```

So condition order is very important.

---

# Correct Order for Ranges

Correct:

```python
when(
    col("amount") >= 50000,
    "HIGH"
)

.when(
    col("amount") >= 30000,
    "MEDIUM"
)

.otherwise(
    "LOW"
)
```

Incorrect order:

```python
when(
    col("amount") >= 30000,
    "MEDIUM"
)

.when(
    col("amount") >= 50000,
    "HIGH"
)
```

Why?

Because:

```text
amount = 60000
```

matches:

```text
amount >= 30000
```

first.

So Spark would return:

```text
MEDIUM
```

and never reach the HIGH condition.

Important rule:

```text
For overlapping conditions,
put the most specific / highest-priority condition first.
```

---

# Multiple Conditions Using AND

Suppose requirement:

```text
amount >= 50000

AND

status = SUCCESS

→ PREMIUM
```

Code:

```python
df2 = df.withColumn(
    "customer_type",

    when(
        (col("amount") >= 50000)
        &
        (col("status") == "SUCCESS"),
        "PREMIUM"
    )

    .otherwise(
        "REGULAR"
    )
)
```

---

# Multiple Conditions Using OR

Requirement:

```text
status = FAILED
OR
status = CANCELLED

→ ERROR
```

Code:

```python
df2 = df.withColumn(
    "status_group",

    when(
        (col("status") == "FAILED")
        |
        (col("status") == "CANCELLED"),
        "ERROR"
    )

    .otherwise(
        "VALID"
    )
)
```

---

# Using `isin()` Inside `when()`

Instead of writing multiple OR conditions:

```python
when(
    (col("status") == "FAILED")
    |
    (col("status") == "CANCELLED"),
    "ERROR"
)
```

we can write:

```python
when(
    col("status").isin(
        "FAILED",
        "CANCELLED"
    ),
    "ERROR"
)
```

This is cleaner.

Full example:

```python
df2 = df.withColumn(
    "status_group",

    when(
        col("status").isin(
            "FAILED",
            "CANCELLED"
        ),
        "ERROR"
    )

    .otherwise(
        "VALID"
    )
)
```

---

# Using `between()` Inside `when()`

Requirement:

```text
amount between 30000 and 49999
→ MEDIUM
```

Code:

```python
df2 = df.withColumn(
    "amount_category",

    when(
        col("amount").between(
            30000,
            49999
        ),
        "MEDIUM"
    )

    .otherwise(
        "OTHER"
    )
)
```

---

# Complete Range Example

```python
df2 = df.withColumn(
    "amount_category",

    when(
        col("amount") >= 50000,
        "HIGH"
    )

    .when(
        col("amount").between(
            30000,
            49999
        ),
        "MEDIUM"
    )

    .otherwise(
        "LOW"
    )
)
```

---

# Using NULL Conditions in `when()`

Suppose amount can be NULL.

Requirement:

```text
amount is NULL
→ UNKNOWN

amount >= 50000
→ HIGH

otherwise
→ NORMAL
```

Code:

```python
df2 = df.withColumn(
    "amount_status",

    when(
        col("amount").isNull(),
        "UNKNOWN"
    )

    .when(
        col("amount") >= 50000,
        "HIGH"
    )

    .otherwise(
        "NORMAL"
    )
)
```

---

# Why Handle NULL First?

Suppose:

```text
amount = NULL
```

Condition:

```python
col("amount") >= 50000
```

does not return TRUE.

It evaluates to UNKNOWN.

So if we need a specific category for NULL values, explicitly check:

```python
col("amount").isNull()
```

---

# Complex Business Rule Example

Requirement:

```text
If amount is NULL
    → INVALID

Else if amount >= 50000 and status = SUCCESS
    → PREMIUM_SUCCESS

Else if amount >= 50000
    → HIGH_VALUE

Else if amount between 30000 and 49999
    → MEDIUM_VALUE

Else
    → LOW_VALUE
```

Code:

```python
df2 = df.withColumn(
    "transaction_category",

    when(
        col("amount").isNull(),
        "INVALID"
    )

    .when(
        (col("amount") >= 50000)
        &
        (col("status") == "SUCCESS"),
        "PREMIUM_SUCCESS"
    )

    .when(
        col("amount") >= 50000,
        "HIGH_VALUE"
    )

    .when(
        col("amount").between(
            30000,
            49999
        ),
        "MEDIUM_VALUE"
    )

    .otherwise(
        "LOW_VALUE"
    )
)
```

---

# Multiple Output Values

`when()` can return different types of expressions, but the final resulting column should have a compatible Spark datatype.

Example:

```python
df2 = df.withColumn(
    "bonus",

    when(
        col("amount") >= 50000,
        col("amount") * 0.10
    )

    .otherwise(
        col("amount") * 0.05
    )
)
```

Here both branches return numeric expressions.

---

# Calculated Values Inside `when()`

Example:

```python
df2 = df.withColumn(
    "discount",

    when(
        col("amount") >= 50000,
        col("amount") * 0.20
    )

    .when(
        col("amount") >= 30000,
        col("amount") * 0.10
    )

    .otherwise(
        col("amount") * 0.05
    )
)
```

Meaning:

```text
>= 50000
→ 20% discount

>= 30000
→ 10% discount

otherwise
→ 5% discount
```

---

# Using `lit()` With `when()`

When returning constant values, Spark often accepts Python literals directly:

```python
when(
    col("amount") >= 50000,
    "HIGH"
)
```

We can also explicitly use:

```python
from pyspark.sql.functions import lit
```

Example:

```python
when(
    col("amount") >= 50000,
    lit("HIGH")
)
```

Both styles are commonly used.

---

# `when()` Inside `select()`

`when()` is not limited to `withColumn()`.

Example:

```python
df.select(
    "customer_id",
    "amount",

    when(
        col("amount") >= 50000,
        "HIGH"
    )
    .otherwise(
        "NORMAL"
    )
    .alias(
        "amount_category"
    )
).show()
```

This creates the calculated expression only in the selected output.

---

# `when()` With `alias()`

Example:

```python
df.select(
    col("customer_id"),

    when(
        col("amount") >= 50000,
        "HIGH"
    )
    .otherwise(
        "LOW"
    )
    .alias(
        "amount_category"
    )
)
```

---

# `when()` vs Python `if`

This is very important.

Wrong:

```python
if col("amount") >= 50000:
    ...
```

Why?

Because:

```python
col("amount")
```

is a Spark Column expression.

It is not a single Python value.

Spark must evaluate the expression for every row across distributed data.

Use:

```python
when(
    col("amount") >= 50000,
    "HIGH"
)
```

---

# Python `if` vs Spark `when()`

```text
Python if
→ Evaluates normal Python values
→ Driver-side logic


Spark when()
→ Evaluates DataFrame columns
→ Row-level distributed logic
```

---

# `when()` Is Not an Action

Example:

```python
df2 = df.withColumn(
    "category",

    when(
        col("amount") >= 50000,
        "HIGH"
    ).otherwise(
        "LOW"
    )
)
```

This does not immediately execute the Spark job.

It creates a new DataFrame logical plan.

Execution happens when:

```python
df2.show()
```

or:

```python
df2.count()
```

or:

```python
df2.write...
```

---

# Spark Execution Concept

```text
Original DataFrame
       ↓
withColumn()
       ↓
when()
       ↓
Conditional Expression
       ↓
Logical Plan
       ↓
Catalyst Optimizer
       ↓
Physical Plan
       ↓
Action
       ↓
Execution
```

---

# Real Project Example 1: Transaction Risk Classification

Requirement:

```text
amount >= 100000
→ HIGH_RISK

amount >= 50000
→ MEDIUM_RISK

otherwise
→ LOW_RISK
```

Code:

```python
df = df.withColumn(
    "risk_level",

    when(
        col("amount") >= 100000,
        "HIGH_RISK"
    )

    .when(
        col("amount") >= 50000,
        "MEDIUM_RISK"
    )

    .otherwise(
        "LOW_RISK"
    )
)
```

---

# Real Project Example 2: Data Quality Flag

Requirement:

```text
customer_id NULL
→ INVALID_CUSTOMER

amount NULL
→ INVALID_AMOUNT

otherwise
→ VALID
```

Code:

```python
df = df.withColumn(
    "dq_status",

    when(
        col("customer_id").isNull(),
        "INVALID_CUSTOMER"
    )

    .when(
        col("amount").isNull(),
        "INVALID_AMOUNT"
    )

    .otherwise(
        "VALID"
    )
)
```

---

# Real Project Example 3: Order Status Classification

Requirement:

```text
SUCCESS
→ COMPLETED

PENDING or PROCESSING
→ IN_PROGRESS

FAILED or CANCELLED
→ UNSUCCESSFUL

anything else
→ UNKNOWN
```

Code:

```python
df = df.withColumn(
    "status_category",

    when(
        col("status") == "SUCCESS",
        "COMPLETED"
    )

    .when(
        col("status").isin(
            "PENDING",
            "PROCESSING"
        ),
        "IN_PROGRESS"
    )

    .when(
        col("status").isin(
            "FAILED",
            "CANCELLED"
        ),
        "UNSUCCESSFUL"
    )

    .otherwise(
        "UNKNOWN"
    )
)
```

---

# Real Project Example 4: Bonus Calculation

Requirement:

```text
amount >= 50000
AND
status = SUCCESS
→ 20% bonus

amount >= 30000
AND
status = SUCCESS
→ 10% bonus

otherwise
→ 0
```

Code:

```python
df = df.withColumn(
    "bonus",

    when(
        (col("amount") >= 50000)
        &
        (col("status") == "SUCCESS"),

        col("amount") * 0.20
    )

    .when(
        (col("amount") >= 30000)
        &
        (col("status") == "SUCCESS"),

        col("amount") * 0.10
    )

    .otherwise(
        0
    )
)
```

---

# Important Rule: Order of Conditions

Suppose:

```python
when(
    col("amount") >= 30000,
    "MEDIUM"
)

.when(
    col("amount") >= 50000,
    "HIGH"
)
```

For:

```text
amount = 60000
```

Spark checks:

```text
60000 >= 30000
→ TRUE
```

So result becomes:

```text
MEDIUM
```

Spark does not continue to later conditions once a condition matches.

Correct ordering:

```python
when(
    col("amount") >= 50000,
    "HIGH"
)

.when(
    col("amount") >= 30000,
    "MEDIUM"
)

.otherwise(
    "LOW"
)
```

---

# Important Rule: First Match Wins

Remember:

```text
Spark evaluates when() conditions
from top to bottom.

The first TRUE condition wins.
```

This is one of the most important interview points.

---

# Common Mistake 1: Missing `otherwise()`

Example:

```python
when(
    col("amount") >= 50000,
    "HIGH"
)
```

Rows that do not match become:

```text
NULL
```

If this is not desired, add:

```python
.otherwise(
    "LOW"
)
```

---

# Common Mistake 2: Wrong Condition Order

Wrong:

```python
when(
    col("amount") >= 30000,
    "MEDIUM"
)

.when(
    col("amount") >= 50000,
    "HIGH"
)
```

Correct:

```python
when(
    col("amount") >= 50000,
    "HIGH"
)

.when(
    col("amount") >= 30000,
    "MEDIUM"
)
```

---

# Common Mistake 3: Using Python `and`

Wrong:

```python
when(
    (col("amount") >= 50000)
    and
    (col("status") == "SUCCESS"),
    "HIGH"
)
```

Correct:

```python
when(
    (col("amount") >= 50000)
    &
    (col("status") == "SUCCESS"),
    "HIGH"
)
```

Use:

```text
& → AND
| → OR
~ → NOT
```

---

# Common Mistake 4: Not Using Parentheses

Avoid:

```python
col("amount") >= 50000 & col("status") == "SUCCESS"
```

Correct:

```python
(
    col("amount") >= 50000
)
&
(
    col("status") == "SUCCESS"
)
```

---

# Common Mistake 5: Ignoring NULL

Suppose:

```text
amount = NULL
```

This:

```python
col("amount") >= 50000
```

does not become TRUE or FALSE in the usual sense.

If NULL requires special business handling, explicitly write:

```python
when(
    col("amount").isNull(),
    "UNKNOWN"
)
```

---

# `when()` vs SQL CASE WHEN

PySpark:

```python
df.withColumn(
    "category",

    when(
        col("amount") >= 50000,
        "HIGH"
    )

    .when(
        col("amount") >= 30000,
        "MEDIUM"
    )

    .otherwise(
        "LOW"
    )
)
```

SQL equivalent:

```sql
CASE
    WHEN amount >= 50000 THEN 'HIGH'
    WHEN amount >= 30000 THEN 'MEDIUM'
    ELSE 'LOW'
END
```

This is useful when explaining `when()` in interviews.

---

# INTERVIEW QUESTIONS

## 1. What is `when()` in PySpark?

`when()` is used to create conditional column expressions. It is similar to SQL `CASE WHEN` or an IF condition.

---

## 2. What is `otherwise()`?

`otherwise()` defines the default result when none of the preceding `when()` conditions are satisfied.

It is equivalent to SQL `ELSE`.

---

## 3. What happens if `otherwise()` is not used?

Rows that do not match any `when()` condition generally receive NULL for that expression.

---

## 4. Can we use multiple `when()` calls?

Yes.

Example:

```python
when(
    condition1,
    value1
)

.when(
    condition2,
    value2
)

.otherwise(
    default_value
)
```

---

## 5. How are multiple conditions evaluated?

They are evaluated from top to bottom.

The first matching condition wins.

---

## 6. Why is condition order important?

Because overlapping conditions may both evaluate to TRUE, but Spark returns the result from the first matching condition.

---

## 7. How do you use AND inside `when()`?

```python
when(
    (condition1)
    &
    (condition2),
    value
)
```

---

## 8. How do you use OR inside `when()`?

```python
when(
    (condition1)
    |
    (condition2),
    value
)
```

---

## 9. Can `when()` return calculated values?

Yes.

Example:

```python
when(
    col("amount") >= 50000,
    col("amount") * 0.20
)
```

---

## 10. Can `when()` be used inside `select()`?

Yes.

Example:

```python
df.select(
    when(
        col("amount") >= 50000,
        "HIGH"
    )
    .otherwise(
        "LOW"
    )
    .alias(
        "category"
    )
)
```

---

## 11. Is `when()` an action?

No.

It creates a column expression and follows Spark's lazy evaluation.

---

## 12. Difference between Python `if` and Spark `when()`?

```text
Python if
→ Driver-side Python logic
→ Works with individual Python values


Spark when()
→ DataFrame expression
→ Evaluated row-by-row across distributed data
```

---

# QUICK REVISION TABLE

| Operation | Purpose |
|---|---|
| `when()` | Define conditional result |
| `otherwise()` | Define default/ELSE result |
| `.when()` | Add another condition |
| `&` | AND |
| `\|` | OR |
| `~` | NOT |
| `isin()` | Check multiple possible values |
| `between()` | Check inclusive range |
| `isNull()` | Handle NULL condition |
| `isNotNull()` | Check non-NULL condition |

---

# MEMORY MAP

```text
6. CONDITIONAL LOGIC
│
├── when()
│   │
│   ├── IF condition
│   └── Return value
│
├── otherwise()
│   │
│   └── ELSE / default value
│
└── Multiple Conditions
    │
    ├── when(condition1, value1)
    │
    ├── .when(condition2, value2)
    │
    ├── .when(condition3, value3)
    │
    └── .otherwise(default)
```

---

# COMPLETE PATTERN

```python
df = df.withColumn(
    "new_column",

    when(
        condition1,
        value1
    )

    .when(
        condition2,
        value2
    )

    .when(
        condition3,
        value3
    )

    .otherwise(
        default_value
    )
)
```

---

# COMPLETE BUSINESS EXAMPLE

Requirement:

```text
1. If amount is NULL
   → INVALID

2. If amount >= 50000 and status = SUCCESS
   → PREMIUM

3. If amount >= 50000
   → HIGH_VALUE

4. If amount between 30000 and 49999
   → MEDIUM_VALUE

5. Otherwise
   → LOW_VALUE
```

Code:

```python
df = df.withColumn(
    "transaction_type",

    when(
        col("amount").isNull(),
        "INVALID"
    )

    .when(
        (col("amount") >= 50000)
        &
        (col("status") == "SUCCESS"),
        "PREMIUM"
    )

    .when(
        col("amount") >= 50000,
        "HIGH_VALUE"
    )

    .when(
        col("amount").between(
            30000,
            49999
        ),
        "MEDIUM_VALUE"
    )

    .otherwise(
        "LOW_VALUE"
    )
)
```

Conceptual execution:

```text
Each Row
   ↓
Check amount IS NULL?
   ↓ No
Check amount >= 50000 AND SUCCESS?
   ↓ No
Check amount >= 50000?
   ↓ No
Check amount between 30000 and 49999?
   ↓ No
otherwise
   ↓
LOW_VALUE
```

Remember:

```text
FIRST MATCH WINS
```

select()
--------------------------------------------------

Purpose:
Select required columns or expressions.

Type:
Transformation

Syntax:

df.select("col1", "col2")

Examples:

df.select("customer_id")

df.select(
    "customer_id",
    "amount"
)

Using col():

df.select(
    col("customer_id"),
    col("amount")
)

All columns:

df.select("*")

Dynamic selection:

cols = [
    "customer_id",
    "product",
    "amount"
]

df.select(*cols)

Calculated column:

df.select(
    "customer_id",
    (col("amount") * 2).alias("double_amount")
)

Important:
- Returns a new DataFrame
- Does not modify original DataFrame
- Can reorder columns
- Can select expressions
- Supports dynamic column lists
- Can help projection pruning
- Does not itself trigger Spark execution

In [ ]:
df1=df.select("customer_id","amount")

In [ ]:
df1.show()

In [ ]:
from pyspark.sql.functions import col 
df.select(
    col("customer_id"),
    col("amount")*2
).show()

col()

col() is used to reference a DataFrame column as a Spark Column object.

from pyspark.sql.functions import col


In [ ]:
data = [
    ("C101", "Laptop", 50000),
    ("C102", "Mobile", 20000),
    ("C103", "TV", 30000)
]

df = spark.createDataFrame(
    data,
    ["customer_id", "product", "amount"]
)

In [ ]:
from pyspark.sql.functions import col
df.select("amount").show()

In [ ]:
df.select(col("amount")).show()

### The important diffrence is that col("amount") gives Spark a Column object, which we cqn use in expression 

#### Why do use col()- 
- String are mainly good for naming column sbut when we want to do operations like- 

    - amount*2
    - amount+1000
    


In [ ]:
df.select(
    col("amount")*2
).show()

In [ ]:
df.select(
    col("customer_id"),
    col("amount"),
    (col("amount")*1.18).alias("Amount with tax")
).show()

In [ ]:
df.filter(
col("amount")>25000
).show()

In [ ]:
df.filter(
    (col("amount")>20000)&
    (col("product")!="TV")
).show()

In [ ]:
# col() with withColumn() 

df2=df.withColumn(
    "double_amount",
    col("amount")*2
)

In [ ]:
df2.show()

In [ ]:
df.printSchema()

In [ ]:
df2.printSchema()

In [ ]:
# alias: It is used to give a temporary name to a column or dataframe expression 

df.select(
    (col("amount")*2).alias("double_amount")
).show()

withColumn() - It is used to add a new column or replace an existing column in a datframe 

withColumn(
    "new_column_name,
    expression
)


Why do we use withcolumn 

- adding dervied columns 
- modifying exsting columns 
- type casting 
- cleaning data 
- applying business ruls 
- creating flag 
- handling a null 
- data transformation 



In [ ]:
df2=df.withColumn(
    "double_amount",
    col("amount")*2
)
df2.show()



In [ ]:
df2.withColumn(
    "tax",
    col("amount")*0.18
).show()

In [ ]:
## Replace an Existing Column 

## If the column name is already exists withColumn() replace it 

df2=df.withColumn(
    "amount",
    col("amount")*2
)

In [ ]:
df2.show()

In [ ]:
## withColumn() with lit() 

from pyspark.sql.functions import lit 

df2=df.withColumn(
    "country",
    lit("India")
)

In [ ]:
df2.show()

In [ ]:
### withColumnRenamed()

# it is used to rename an existing column in a dataframe 

df.withColumnRenamed(
    "amount",
    "xxx_amount"
).show()



In [ ]:
## drop() - It is used to remove one or more columns from DF 

df3=df2.drop("country")

In [ ]:
df3.show()

├── 5. Filtering
│   ├── filter()
│   ├── where()
│   ├── AND / OR / NOT
│   ├── isin()
│   ├── between()
│   └── NULL handling

data=[
    (1,"laptop",50000,2),
    (2,"Phone",30000,3),
    (3,"Tablet",20000,5)

]

columns=[pid,pname,,price,quantity]total_amount=price*qnatity



Tssk:

using wiithColumn creat:
